In [3]:
import json
import os

with open('output_human/groundingLMM/preds.jsonl') as f:
    preds = [json.loads(line) for line in f]
with open('/home/v-jinjzhao/datasets/rec_human_celeb_ready/person_annts_val_ready.json') as f:
    all_annts= json.load(f)

In [4]:
print(len(preds), len(all_annts))

44738 44738


In [6]:
all_gt_bboxes=[]
all_pred_bboxes=[]
for annt, pred in zip(all_annts, preds):
    assert annt['annt_id'] == pred['annt_id']
    if(annt['recognition_category']!='celebrity'):
        x,y,w,h=annt['bbox']
        all_gt_bboxes.append([x,y,x+w,y+h])
    else:
        all_gt_bboxes.append(annt['bbox'])
    all_pred_bboxes.append(pred['pred_bboxes'][0])

In [7]:
import json
import torch
from tqdm import tqdm
from overlaps import bbox_overlaps
# import average from math
from statistics import mean


def calculate_iou_acc(pred_bboxes,gt_bboxes, thresh=0.5):
    """
    pred_bboxes: [N,4]
    gt_bboxes: [N,4]
    calculate the iou and acc of the pred_bboxes and gt_bboxes,
    if iou(pred_bboxes[i],gt_bboxes[i])>0.5, then acc+=1
    all pred_bboxes_i and gt_bboxes_i are one to one assigned.
    
    """
    iou=bbox_overlaps(pred_bboxes,gt_bboxes,mode='iou', is_aligned=True)
    if(type(thresh) is not list):
        thresh=[thresh]
    accs=dict()
    for t in thresh:
        accs[t]=(iou>t).sum().item()/len(iou)
    return iou,accs




pred_bboxes=torch.tensor(all_pred_bboxes)
gt_bboxes=torch.tensor(all_gt_bboxes)
# create a list fron 0.5 to 0.9 with step 0.05
thresh=[i/100 for i in range(50,95,5)]
print(thresh)
iou,acc=calculate_iou_acc(pred_bboxes,gt_bboxes,thresh)
print_ths=[0.5,0.7,0.9]
for k,v in acc.items():
    if(k in print_ths):
        print(f"thresh={k}, acc={v}")
print(f"iou|0.5:0.9={mean([v for v in acc.values()])}")

print(iou)
print(acc)
print(f"{len(iou)}/{len(all_annts)}")

[0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
thresh=0.5, acc=0.6613393535696723
thresh=0.7, acc=0.591264696678439
thresh=0.9, acc=0.44163798113460595
iou|0.5:0.9=0.5772870192379335
tensor([0.9273, 0.8245, 0.9348,  ..., 0.5250, 0.9035, 0.0276])
{0.5: 0.6613393535696723, 0.55: 0.6451338906522419, 0.6: 0.627564933613483, 0.65: 0.610152443113237, 0.7: 0.591264696678439, 0.75: 0.5688229245831284, 0.8: 0.5433188788054898, 0.85: 0.5063480709911038, 0.9: 0.44163798113460595}
44738/44738


In [8]:
for pred_i,iou_i, annt in zip(pred_bboxes,iou,all_annts):
    annt['groundingLMM_pred']=dict(
        pred_bbox=pred_i.tolist(),
        iou=iou_i.item()
    )

In [9]:
all_annts[-1]

{'annt_id': '0044737',
 'bbox': [196.128, 153.90599999999998, 840.3539999999999, 667.38],
 'bbox_area': 330793.301124,
 'caption': 'Zack Wheeler',
 'category_id': 1,
 'file_name': 'Zack_Wheeler_0180.jpg',
 'height': 681,
 'image_id': 'Zack_Wheeler_0180',
 'recognition_category': 'celebrity',
 'width': 1024,
 'source': 'web',
 'original_subset': 'val',
 'is_rewrite': False,
 'groundingGPT_pred': {'pred_bboxes': [256.0,
   115.72000122070312,
   778.239990234375,
   669.0],
  'iou': 0.46317657828330994},
 'shikra7b_pred': {'iou': 0.7647262811660767,
  'pred_bbox': [261.1199951171875,
   128.00799560546875,
   785.4080200195312,
   681.9920043945312]},
 'Lenna_pred': {'pred_bboxes': [170.35012817382812,
   126.15596771240234,
   758.0120849609375,
   672.5585327148438],
  'iou': 0.7939690351486206},
 'PSALM_pred': {'pred_bboxes': [164, 137, 335, 436],
  'iou': 0.11429689824581146},
 'groundingLMM_pred': {'pred_bbox': [6, 430, 240, 671],
  'iou': 0.027640873566269875}}

In [10]:
with open('/home/v-jinjzhao/datasets/rec_human_celeb_ready/person_annts_val_ready.json','w') as f:
    json.dump(all_annts,f,indent=4)